In [1]:
!pip install --upgrade pip -q
!pip install ultralytics ipywidgets --upgrade  -q
!pip install torch --index-url https://download.pytorch.org/whl/cu130 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 32.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.


In [2]:
import os
from pathlib import Path

import torch
import yaml
from ultralytics import YOLO

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
PyTorch version: 2.9.0+cu126
CUDA available: True
CUDA device: Tesla T4


In [3]:
# Paths
DATASET_PATH = Path("/kaggle/input/datasets/rayraurray/ingredients-dataset/ingredients_dataset")
# DATASET_PATH = "/kaggle/input/datasets/rayraurray/ingredients-dataset/ingredients_dataset"
OG_YAML = DATASET_PATH / "data.yaml"
DATA_YAML = "/kaggle/working/data_fixed.yaml"

WEIGHTS_DIR = Path("weights")
WEIGHTS_DIR.mkdir(exist_ok=True)

In [4]:
with open(OG_YAML, "r") as f:
    data = yaml.safe_load(f)

data["path"] = str(DATASET_PATH)

with open(DATA_YAML, "w") as f:
    yaml.safe_dump(data, f, sort_keys=False)

print("Fixed YAML written to", DATA_YAML)

Fixed YAML written to /kaggle/working/data_fixed.yaml


In [5]:
# Training hyperparameters
CONFIG = {
    "model": "yolo11n.pt",  # Options: yolo11n, yolo11s, yolo11m, yolo11l, yolo11x
    "epochs": 100,
    "imgsz": 640,
    "batch": 16,
    "device": 0 if torch.cuda.is_available() else "cpu",
    "workers": 8,
    "patience": 50, # Early stopping patience
    "save": True,
    "save_period": 10, # Save checkpoint every N epochs
    "cache": False, # Cache images for faster training (use True if enough RAM)
    "optimizer": "auto", # Options: SGD, Adam, AdamW, NAdam, RAdam, RMSProp, auto
    "lr0": 0.01, # Initial learning rate
    "lrf": 0.01, # Final learning rate (lr0 * lrf)
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "warmup_epochs": 3.0,
    "warmup_momentum": 0.8,
    "warmup_bias_lr": 0.1,
    "box": 7.5,  # Box loss gain
    "cls": 0.5,  # Class loss gain
    "dfl": 1.5,  # DFL loss gain
    "hsv_h": 0.015,  # HSV-Hue augmentation
    "hsv_s": 0.7,  # HSV-Saturation augmentation
    "hsv_v": 0.4,  # HSV-Value augmentation
    "degrees": 0.0,  # Rotation augmentation
    "translate": 0.1,  # Translation augmentation
    "scale": 0.5,  # Scale augmentation
    "shear": 0.0,  # Shear augmentation
    "perspective": 0.0,  # Perspective augmentation
    "flipud": 0.0,  # Vertical flip probability
    "fliplr": 0.5,  # Horizontal flip probability
    "mosaic": 1.0,  # Mosaic augmentation probability
    "mixup": 0.0,  # Mixup augmentation probability
    "copy_paste": 0.0,  # Copy-paste augmentation probability
}

In [6]:
print("Configuration:")
for key, value in CONFIG.items():
    print(f"{key}: {value}")

Configuration:
model: yolo11n.pt
epochs: 100
imgsz: 640
batch: 16
device: 0
workers: 8
patience: 50
save: True
save_period: 10
cache: False
optimizer: auto
lr0: 0.01
lrf: 0.01
momentum: 0.937
weight_decay: 0.0005
warmup_epochs: 3.0
warmup_momentum: 0.8
warmup_bias_lr: 0.1
box: 7.5
cls: 0.5
dfl: 1.5
hsv_h: 0.015
hsv_s: 0.7
hsv_v: 0.4
degrees: 0.0
translate: 0.1
scale: 0.5
shear: 0.0
perspective: 0.0
flipud: 0.0
fliplr: 0.5
mosaic: 1.0
mixup: 0.0
copy_paste: 0.0


In [7]:
print(f"\nVerifying dataset at: {DATA_YAML}")

# Load and display dataset info
with open(DATA_YAML, 'r') as f:
    data_config = yaml.safe_load(f)

print(f"Number of classes: {data_config['nc']}")
print(f"Train path: {data_config['train']}")
print(f"Val path: {data_config['val']}")
print(f"Test path: {data_config['test']}")
print(f"\nFirst 10 classes: {data_config['names'][:10]}")
print(f"Last 10 classes: {data_config['names'][-10:]}")

# Check if directories exist
train_images = Path(data_config['path']) / data_config['train']
val_images = Path(data_config['path']) / data_config['val']
test_images = Path(data_config['path']) / data_config['test']

print(f"\nTrain images exist: {train_images.exists()}")
print(f"Val images exist: {val_images.exists()}")
print(f"Test images exist: {test_images.exists()}")


Verifying dataset at: /kaggle/working/data_fixed.yaml
Number of classes: 207
Train path: train/images
Val path: val/images
Test path: test/images

First 10 classes: ['agar agar powder', 'alsa powder', 'annatto oil', 'bamboo shoot', 'banana', 'banana blossom', 'banana leaf', 'basil', 'bean sprout', 'beef ball']
Last 10 classes: ['vanilla extract', 'water chestnut', 'water spinach', 'watercress', 'wheat flour', 'wine', 'winter melon', 'wonton', 'wood ear mushroom', 'yeast']

Train images exist: True
Val images exist: True
Test images exist: True


In [8]:
print(f"\nInitializing YOLOv11 model: {CONFIG['model']}")

# Load pretrained model
model = YOLO(CONFIG['model'])

# Display model info
print(f"Model loaded successfully")
print(f"Model type: {type(model)}")


Initializing YOLOv11 model: yolo11n.pt
Model loaded successfully
Model type: <class 'ultralytics.models.yolo.model.YOLO'>


In [9]:
print("\n" + "="*80)
print("Starting Training")
print("="*80 + "\n")

# Train the model
results = model.train(
    data=str(DATA_YAML),
    epochs=CONFIG['epochs'],
    imgsz=CONFIG['imgsz'],
    batch=CONFIG['batch'],
    device=CONFIG['device'],
    workers=CONFIG['workers'],
    patience=CONFIG['patience'],
    save=CONFIG['save'],
    save_period=CONFIG['save_period'],
    cache=CONFIG['cache'],
    optimizer=CONFIG['optimizer'],
    lr0=CONFIG['lr0'],
    lrf=CONFIG['lrf'],
    momentum=CONFIG['momentum'],
    weight_decay=CONFIG['weight_decay'],
    warmup_epochs=CONFIG['warmup_epochs'],
    warmup_momentum=CONFIG['warmup_momentum'],
    warmup_bias_lr=CONFIG['warmup_bias_lr'],
    box=CONFIG['box'],
    cls=CONFIG['cls'],
    dfl=CONFIG['dfl'],
    hsv_h=CONFIG['hsv_h'],
    hsv_s=CONFIG['hsv_s'],
    hsv_v=CONFIG['hsv_v'],
    degrees=CONFIG['degrees'],
    translate=CONFIG['translate'],
    scale=CONFIG['scale'],
    shear=CONFIG['shear'],
    perspective=CONFIG['perspective'],
    flipud=CONFIG['flipud'],
    fliplr=CONFIG['fliplr'],
    mosaic=CONFIG['mosaic'],
    mixup=CONFIG['mixup'],
    copy_paste=CONFIG['copy_paste'],
    project=str(WEIGHTS_DIR),
    name='ingredient_detection',
    exist_ok=True,
    pretrained=True,
    verbose=True,
)

print("\n" + "="*80)
print("Training Complete!")
print("="*80)


Starting Training

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data_fixed.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=ingredient_detection, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mas

libpng warning: eXIf: duplicate


val: Scanning /kaggle/input/datasets/rayraurray/ingredients-dataset/ingredients_dataset/val/labels... 650 images, 0 backgrounds, 1 corrupt: 100% ━━━━━━━━━━━━ 651/651 246.8it/s 2.6s
val: /kaggle/input/datasets/rayraurray/ingredients-dataset/ingredients_dataset/val/images/tomato_image_16.jpg: ignoring corrupt image/label: [Errno 30] Read-only file system: '/kaggle/input/datasets/rayraurray/ingredients-dataset/ingredients_dataset/val/images/tomato_image_16.jpg'
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/rayraurray/ingredients-dataset/ingredients_dataset/val is not writable, cache not saved.
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=4.7e-05, momentum=0.9) with parameter groups 81 weight(decay=0.0), 88 weight(decay=0.0005), 87 bias(decay=0.0)
Plotting labels to /kaggle/working/runs/detect/weights/ingredient_detection/labels.jpg... 
Image sizes 640 train,

libpng warning: iCCP: extra compressed data


      1/100         3G     0.7308      5.213      1.264        117        640: 90% ━━━━━━━━━━╸─ 215/239 4.8it/s 1:03<5.0s

...

    100/100      3.04G     0.3419      1.496     0.9542         32        640: 82% ━━━━━━━━━╸── 195/239 4.4it/s 43.7s<10.0s

libpng warning: eXIf: duplicate


    100/100      3.04G     0.3428        1.5     0.9562         24        640: 87% ━━━━━━━━━━── 209/239 4.6it/s 47.1s<6.5s

libpng warning: iCCP: extra compressed data


    100/100      3.04G      0.342      1.498     0.9586         21        640: 100% ━━━━━━━━━━━━ 239/239 4.4it/s 54.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 4.7it/s 4.5s
                   all        650       1409       0.53      0.577      0.581      0.524

100 epochs completed in 1.720 hours.
Optimizer stripped from /kaggle/working/runs/detect/weights/ingredient_detection/weights/last.pt, 5.7MB
Optimizer stripped from /kaggle/working/runs/detect/weights/ingredient_detection/weights/best.pt, 5.7MB

Validating /kaggle/working/runs/detect/weights/ingredient_detection/weights/best.pt...
Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,680,009 parameters, 0 gradients, 6.8 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 4.2it/s 5.1s
                   all        650      

In [10]:
print("\n" + "="*80)
print("Running Validation")
print("="*80 + "\n")

# Validate the model
metrics = model.val()

print(f"\nValidation Results:")
print(f"  mAP50: {metrics.box.map50:.4f}")
print(f"  mAP50-95: {metrics.box.map:.4f}")
print(f"  Precision: {metrics.box.mp:.4f}")
print(f"  Recall: {metrics.box.mr:.4f}")


Running Validation

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,680,009 parameters, 0 gradients, 6.8 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 250.9±334.7 MB/s, size: 185.7 KB)
val: Scanning /kaggle/input/datasets/rayraurray/ingredients-dataset/ingredients_dataset/val/labels... 650 images, 0 backgrounds, 1 corrupt: 100% ━━━━━━━━━━━━ 651/651 612.8it/s 1.1s
val: /kaggle/input/datasets/rayraurray/ingredients-dataset/ingredients_dataset/val/images/tomato_image_16.jpg: ignoring corrupt image/label: [Errno 30] Read-only file system: '/kaggle/input/datasets/rayraurray/ingredients-dataset/ingredients_dataset/val/images/tomato_image_16.jpg'
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/rayraurray/ingredients-dataset/ingredients_dataset/val is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 41/41 6.2it/s 6

In [11]:
print("\n" + "="*80)
print("Exporting Model")
print("="*80 + "\n")

# Export to different formats
# Uncomment the formats you need

# ONNX format (recommended for deployment)
# model.export(format='onnx')

# TensorRT format (for NVIDIA GPUs)
# model.export(format='engine')

# TorchScript format
# model.export(format='torchscript')

# CoreML format (for iOS)
# model.export(format='coreml')
model.save('best.pt')
# print("Model export skipped. Uncomment desired formats in Cell 7 to export.")


Exporting Model



In [12]:
print("\n" + "="*80)
print("Test Inference")
print("="*80 + "\n")

# Test on a sample image
# Uncomment to run inference on test images

# test_image = "/kaggle/input/datasets/rayraurray/ingredients-dataset/ingredients_dataset/test/images/beef jerky_image_21.jpg"
test_image = "/kaggle/input/datasets/rayraurray/ingredients-dataset/ingredients_dataset/test/images/banana blossom_image_0.jpg"
results = model.predict(
    source=test_image,
    conf=0.25, # Confidence threshold
    iou=0.45, # NMS IoU threshold
    save=True, # Save results
    show=False, # Show results
)

for result in results:
    boxes = result.boxes
    print(f"Detected {len(boxes)} objects")
    for box in boxes:
        cls = int(box.cls[0])
        conf = float(box.conf[0])
        print(f"  Class: {data_config['names'][cls]}, Confidence: {conf:.2f}")




Test Inference


image 1/1 /kaggle/input/datasets/rayraurray/ingredients-dataset/ingredients_dataset/test/images/banana blossom_image_0.jpg: 480x640 4 banana blossoms, 52.1ms
Speed: 2.1ms preprocess, 52.1ms inference, 1.2ms postprocess per image at shape (1, 3, 480, 640)
Results saved to /kaggle/working/runs/detect/predict
Detected 4 objects
  Class: banana blossom, Confidence: 0.92
  Class: banana blossom, Confidence: 0.86
  Class: banana blossom, Confidence: 0.39
  Class: banana blossom, Confidence: 0.27


In [13]:
# ============================================================================
# Cell 9: Summary
# ============================================================================

print("\n" + "="*80)
print("Training Summary")
print("="*80)
print(f"\nModel: {CONFIG['model']}")
print(f"Dataset: {DATA_YAML}")
print(f"Classes: {data_config['nc']}")
print(f"Epochs: {CONFIG['epochs']}")
print(f"Image size: {CONFIG['imgsz']}")
print(f"Batch size: {CONFIG['batch']}")
print(f"Device: {CONFIG['device']}")
print(f"\nBest weights saved to: {WEIGHTS_DIR / 'ingredient_detection' / 'weights' / 'best.pt'}")
print(f"Last weights saved to: {WEIGHTS_DIR / 'ingredient_detection' / 'weights' / 'last.pt'}")
print("\nTraining complete! Check the weights directory for results.")
print("="*80 + "\n")


Training Summary

Model: yolo11n.pt
Dataset: /kaggle/working/data_fixed.yaml
Classes: 207
Epochs: 100
Image size: 640
Batch size: 16
Device: 0

Best weights saved to: weights/ingredient_detection/weights/best.pt
Last weights saved to: weights/ingredient_detection/weights/last.pt

Training complete! Check the weights directory for results.

